# 02 — Data Quality & Cleaning Validation

## Purpose
This notebook validates the **business-driven data cleaning rules** applied in Step 2
(`src/cleaning/clean_transactions.py`).

The goal is to:
- Quantify how much data was removed or corrected
- Demonstrate that removals are justified and not arbitrary
- Confirm that the cleaned dataset is suitable for downstream modeling

## Scope
- This notebook does NOT introduce new cleaning logic
- This notebook does NOT perform feature engineering
- All cleaning logic lives in the production pipeline code

This notebook exists to provide **transparency, auditability, and trust**.

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [3]:
raw_path = "/Users/ramesh/Desktop/clv-long-term-optimization/data/interim/transactions_raw.parquet"
clean_path = "/Users/ramesh/Desktop/clv-long-term-optimization/data/interim/transactions_clean.parquet"

raw = pd.read_parquet(raw_path)
clean = pd.read_parquet(clean_path)

raw.shape, clean.shape

((1067371, 9), (793609, 10))

## High-level impact of cleaning

Before examining individual rules, we compare dataset size and coverage
before and after cleaning to understand the overall impact.

In [4]:
summary = pd.DataFrame(
    {
        "rows": [len(raw), len(clean)],
        "customers": [
            raw["customer_id"].nunique(dropna=True),
            clean["customer_id"].nunique(dropna=True),
        ],
        "invoices": [
            raw["invoice"].nunique(dropna=True),
            clean["invoice"].nunique(dropna=True),
        ],
        "min_date": [raw["invoice_dt"].min(), clean["invoice_dt"].min()],
        "max_date": [raw["invoice_dt"].max(), clean["invoice_dt"].max()],
    },
    index=["raw", "cleaned"],
)

summary

,rows,customers,invoices,min_date,max_date
raw,1067371,5942,53628,2009-12-01 07:45:00,2011-12-09 12:50:00
cleaned,793609,5878,36969,2009-12-01 07:45:00,2011-12-09 12:50:00


## Validation of individual cleaning rules

Each rule below corresponds directly to a documented business rule
implemented in the cleaning pipeline.

For each rule, we quantify:
- How many rows are affected
- Why the rule is necessary
- What risk it mitigates

In [5]:
cancel_rate = raw["invoice"].astype("string").str.startswith("C", na=False).mean()

cancel_rate

np.float64(0.018263565339511754)

**Interpretation**

Invoices starting with `"C"` represent cancellations rather than purchases.
Including them would:
- Artificially inflate frequency
- Introduce negative or reversed revenue
- Break CLV and churn assumptions

Removing these rows is required for valid purchase modeling.

In [6]:
neg_qty_rate = (raw["quantity"].fillna(0) < 0).mean()
nonpos_qty_rate = (raw["quantity"].fillna(0) <= 0).mean()

neg_qty_rate, nonpos_qty_rate

(np.float64(0.021501427338760374), np.float64(0.021501427338760374))

**Interpretation**

Negative or zero quantities correspond to returns, corrections, or data errors.
These do not represent revenue-generating customer behavior and must be excluded.

In [7]:
nonpos_price_rate = (raw["unit_price"].fillna(0) <= 0).mean()

nonpos_price_rate

np.float64(0.005815222635803296)

**Interpretation**

Transactions with zero or negative prices invalidate revenue calculations.
Keeping them would bias monetary value downward and distort CLV estimates.

In [8]:
missing_customer_rate = raw["customer_id"].isna().mean()

missing_customer_rate

np.float64(0.22766872999172733)

**Interpretation**

Transactions without a customer identifier cannot be assigned to a customer.
They are unusable for customer-level analytics and must be removed.

In [9]:
dup_rate = raw.duplicated().mean()

dup_rate

np.float64(0.011367181607894537)

**Interpretation**

Exact duplicate rows artificially inflate purchase frequency and revenue.
Removing duplicates preserves transaction integrity.

In [10]:
clean["revenue"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

count    793609.000000
mean         22.284854
std         225.708594
min           0.001000
50%          12.480000
90%          35.700000
95%          67.500000
99%         203.520000
max      168469.600000
Name: revenue, dtype: float64

**Interpretation**

After cleaning:
- Revenue is strictly positive
- Distribution remains heavy-tailed (expected)
- Values are economically interpretable

This confirms that the cleaned dataset reflects genuine purchase behavior.

## Cleaning validation summary

The applied cleaning rules:
- Remove transactions that do not represent genuine purchases
- Preserve economically meaningful customer behavior
- Reduce noise without over-filtering valuable data

After cleaning:
- All remaining transactions are attributable to customers
- Revenue and frequency metrics are well-defined
- The dataset is suitable for CLV modeling, churn risk estimation, and optimization

This validated dataset is used as the foundation for all downstream modeling steps.